# 02 Random Forest Model

This notebook trains a leakage-safe Random Forest model and compares it with the naive baseline.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_processing import load_gold_data, clean_gold_data
from src.feature_engineering import prepare_modeling_data
from src.model import chronological_train_test_split, train_random_forest_model, calculate_naive_baseline, evaluate_predictions, build_results_dataframe, compare_model_with_baseline, get_feature_importance
from src.prediction import predict_next_trading_day
from src.visualization import plot_actual_vs_predicted, plot_prediction_error, plot_feature_importance

DATA_PATH = PROJECT_ROOT / "data" / "gold_prices.csv"

In [ ]:
raw_gold_data = load_gold_data(DATA_PATH)
clean_data, cleaning_report = clean_gold_data(raw_gold_data)
modeling_data, feature_columns, generated_nan_counts = prepare_modeling_data(clean_data)
feature_columns, generated_nan_counts

In [ ]:
split_data = chronological_train_test_split(modeling_data, feature_columns)
print("Training period:", split_data["training_data"]["Date"].min(), "to", split_data["training_data"]["Date"].max())
print("Testing period:", split_data["testing_data"]["Date"].min(), "to", split_data["testing_data"]["Date"].max())
print("Training observations:", len(split_data["training_data"]))
print("Testing observations:", len(split_data["testing_data"]))

In [ ]:
random_forest_model = train_random_forest_model(split_data["training_features"], split_data["training_target"])
predicted_gold_price = random_forest_model.predict(split_data["testing_features"])
naive_predictions = calculate_naive_baseline(split_data["testing_data"])

In [ ]:
random_forest_metrics = evaluate_predictions(split_data["testing_target"], predicted_gold_price)
baseline_metrics = evaluate_predictions(split_data["testing_target"], naive_predictions)
print("Random Forest:", random_forest_metrics)
print("Naive Baseline:", baseline_metrics)
print(compare_model_with_baseline(baseline_metrics, random_forest_metrics))

In [ ]:
results_data = build_results_dataframe(split_data["testing_data"], predicted_gold_price)
results_data.head()

In [ ]:
feature_importance = get_feature_importance(random_forest_model, feature_columns)
feature_importance

In [ ]:
plot_actual_vs_predicted(results_data)
plot_prediction_error(results_data)
plot_feature_importance(feature_importance)

In [ ]:
predict_next_trading_day(random_forest_model, clean_data, feature_columns)